In [1]:
using CloudAtlas
using LinearAlgebra
using Statistics
using Random
using Dates
using DelimitedFiles
using Serialization
using Base.Threads
using ChannelflowWrapper



SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [5]:

# Domain sizes (α = 2π/Lx, γ = 2π/Lz)
# Choose these to match the target Channelflow DNS box.
α, γ = 1.0, 2.0

Re = 300.0

# Discretizations: (J, K, L)
J, K, L = 2, 4, 7

# Symmetry groups to explore
sx, sy, sz, tx, tz = CloudAtlas.halfbox_symmetries()
H = [(sx * sy) * (tx * tz)]

# Hookstep parameters
hookparams = SearchParams(
    ftol = 1e-8,
    xtol = 1e-10,
    δ = 0.02,
    Nnewton = 20,
    Nhook = 4,
    Nmusearch = 6,
    verbosity = 0,
)

# Guess strategy options: :random, :shear_target, :trajectory
guess_strategy = :random
xnorm = 0.4

# Dedup tolerances
fp_tol = (
    cx = 1e-3,
    cz = 1e-3,
    nm = 2e-2,
    shear = 2e-2,
)

# Accept/reject thresholds
norm_threshold = 1e-3
speed_threshold = 1e-5
promote_norm_threshold = 1e-2
promote_residual_tol = 1e-6

# Channelflow promotion settings
T = 10.0

10.0

In [6]:
model = ODEModel(α, γ, J, K, L, H; normalize = false, tw = true)

J,K,L,m == 2,4,7,338
(2J+1)(2K+1)(2L+1) + 1 == 676
Making matrices B,A1,A2,S3...
Making quadratic operator N...
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 

ODEModel{Float64, LU{Float64, Matrix{Float64}, Vector{Int64}}, CloudAtlas.var"#f#35"{LU{Float64, Matrix{Float64}, Vector{Int64}}, SparseBilinear{Float64}, Matrix{Float64}, Matrix{Float64}}, CloudAtlas.var"#Df#36"{LU{Float64, Matrix{Float64}, Vector{Int64}}, SparseBilinear{Float64}, Matrix{Float64}, Matrix{Float64}}}(1.0, 2.0, Symmetry[Symmetry(-1, -1, 1, 1//2, 1//2, 1)], [1 0 0 1; 1 0 0 3; … ; 6 2 4 5; 6 2 4 7], BasisFunction{Float64}[BasisFunction{Float64}(BasisComponent{Float64}[BasisComponent{Float64}(1.0, FourierMode{Float64}(1.0, 0, 1.0), FourierMode{Float64}(1.0, 0, 2.0), Polynomial(-4.0*y + 4.0*y^3), -1), BasisComponent{Float64}(0.0, FourierMode{Float64}(0.0, 0, 1.0), FourierMode{Float64}(0.0, 0, 2.0), Polynomial(0.0), 0), BasisComponent{Float64}(0.0, FourierMode{Float64}(0.0, 0, 1.0), FourierMode{Float64}(0.0, 0, 2.0), Polynomial(0.0), 0)]), BasisFunction{Float64}(BasisComponent{Float64}[BasisComponent{Float64}(1.0, FourierMode{Float64}(1.0, 0, 1.0), FourierMode{Float64}(1.0, 0

In [9]:
for i = 1:100
    x = randn(length(model))
    println("dot(Cx*x, x) = $(dot(model.Cx*x, x))")
    println("dot(Cz*x, x) = $(dot(model.Cz*x, x))")
end

dot(Cx*x, x) = 0.0
dot(Cz*x, x) = -3.410605131648481e-13
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = -5.115907697472721e-13
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = 0.0
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = -5.115907697472721e-13
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = 8.526512829121202e-14
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = 3.410605131648481e-13
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = -6.927791673660977e-14
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = 1.4992451724538114e-12
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = 1.1368683772161603e-13
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = -1.4210854715202004e-13
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = 1.6342482922482304e-13
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = 1.9895196601282805e-13
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = -3.552713678800501e-13
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = 2.717825964282383e-13
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = -4.405364961712621e-13
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = 1.4921397450962104e-13
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = 8.526512829121202e-14
dot(Cx*x, x) = 0.0
dot(Cz*x, x) = 2.2737367544323206e

In [15]:
Cz = model.Cz
norm(Cz + Cz') / norm(Cz)

0.0

In [18]:
norm(Cz + Cz')

0.0